In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronzelayer LOCATION 'abfss://bronzelayer@policysysadls.dfs.core.windows.net/';
CREATE SCHEMA IF NOT EXISTS silverlayer LOCATION 'abfss://silverlayer@policysysadls.dfs.core.windows.net/';

CREATE TABLE IF NOT EXISTS bronzelayer.Agent USING DELTA LOCATION 'abfss://bronzelayer@policysysadls.dfs.core.windows.net/Agent';
CREATE TABLE IF NOT EXISTS bronzelayer.Claim USING DELTA LOCATION 'abfss://bronzelayer@policysysadls.dfs.core.windows.net/Claim';
CREATE TABLE IF NOT EXISTS bronzelayer.Branch USING DELTA LOCATION 'abfss://bronzelayer@policysysadls.dfs.core.windows.net/Branch';

In [0]:
df=spark.sql("select * from bronzelayer.Agent where merge_flag=false")
display(df)

#Remove all duplicate records based on Agent ID

### Remove all rows where Branch Id does not exist in Branch table and also remove the records if the phone number length is not 10

In [0]:
df_result = spark.sql("""SELECT * FROM (SELECT agent.*, ROW_NUMBER() OVER (PARTITION BY agent.agent_id ORDER BY agent.create_timestamp DESC) as rn FROM bronzelayer.agent INNER JOIN bronzelayer.branch 
    ON agent.branch_id = branch.branch_id  WHERE agent.merge_flag = false and length(agent_phone)=10
) WHERE rn = 1""")
display(df_result)

In [0]:
from pyspark.sql.functions import *
df_phone=df_result.where(length(col("agent_phone"))==10)
display(df_phone)

##Replace all the empty email with 'admin@azurelib.com'

In [0]:
df_email =df_result.fillna({'agent_email':'admin@azurelib.com'})
display(df_email)

In [0]:
display(df_result.filter("agent_id = 2030"))

In [0]:
df_result.createOrReplaceTempView("agent_temp")
df_email =spark.sql("""
    SELECT 
        agent_id,
        agent_name,
        agent_phone,
        branch_id,
        create_timestamp,
        CASE WHEN agent_email = '' OR agent_email IS NULL 
             THEN 'admin@azurelib.com' 
             ELSE agent_email 
        END as agent_email
    FROM agent_temp
""")
display(df_email)

In [0]:
display(df_email.filter("agent_id = 2030"))

Add the merged_date_timestamp (current timesatmp)

In [0]:
#df_final = df_email.withColumn("merged_timestamp",current_timestamp())
df_email.createOrReplaceTempView("clean_agent")
spark.sql("Merge into silverlayer.agent as T USING clean_agent AS S ON t.agent_id = s.agent_id WHEN MATCHED then UPDATE SET t.agent_phone = s.agent_phone, t.agent_email = s.agent_email,t.agent_name = s.agent_name,t.branch_id=s.branch_id,t.create_Timestamp = s.create_TimeStamp, T.merged_timestamp = current_timestamp() When not matched then insert (agent_phone, agent_email ,agent_name ,branch_id,create_Timestamp,merged_timestamp,agent_id) values(s.agent_phone, s.agent_email, s.agent_name,s.branch_id,s.create_TimeStamp,current_timestamp(),s.agent_id)")

In [0]:
display(spark.sql("SELECT * FROM silverlayer.Agent "))

In [0]:
spark.sql("update bronzelayer.agent set merge_flag = True WHERE merge_flag = false")